## Step 0: Import libraries

In [1]:
import os
import shutil
import tempfile
import numpy as np
import pandas as pd

## Step 1: Configuration

In [2]:
TOTAL_ROWS  = 1_000_000     # increase (e.g. 10_000_000) to test a bigger file
WRITE_BATCH = 200_000       # rows written per batch while generating the file
CHUNK_SIZE  = 100_000       # rows read per chunk while processing

rng = np.random.default_rng(42)
tmp_dir  = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, "large_transactions.csv")

## Step 2: Generate the large CSV file (in batches, so generation is memory-safe too)

In [3]:
written = 0
while written < TOTAL_ROWS:
    m = min(WRITE_BATCH, TOTAL_ROWS - written)
    batch = pd.DataFrame({
        "transaction_id": np.arange(written + 1, written + m + 1),
        "customer_id": rng.integers(1, 50_000, m),
        "amount": np.round(rng.gamma(2.0, 60.0, m), 2),
        "city": rng.choice(["Surat", "Mumbai", "Pune", "Delhi", "Rajkot"], m),
    })
    batch.to_csv(csv_path, mode="a", header=(written == 0), index=False)
    written += m

print(f"File created: {os.path.getsize(csv_path) / 1024**2:.1f} MB, {TOTAL_ROWS:,} rows")

File created: 25.0 MB, 1,000,000 rows


## Step 3: Chunked ingestion with running aggregates

In [4]:
total_records = 0
total_amount = 0.0

reader = pd.read_csv(csv_path, usecols=["transaction_id", "amount"], chunksize=CHUNK_SIZE)

print(f"{'Chunk':>5} | {'Rows':>8} | {'Cum. records':>12} | {'Running sum':>16} | {'Running avg':>11}")
print("-" * 66)
for i, chunk in enumerate(reader, start=1):
    total_records += len(chunk)               # cumulative record count
    total_amount  += chunk["amount"].sum()    # running sum
    print(f"{i:>5} | {len(chunk):>8,} | {total_records:>12,} | {total_amount:>16,.2f} | "
          f"{total_amount / total_records:>11.4f}")

Chunk |     Rows | Cum. records |      Running sum | Running avg
------------------------------------------------------------------
    1 |  100,000 |      100,000 |    12,001,916.45 |    120.0192
    2 |  100,000 |      200,000 |    24,006,389.14 |    120.0319
    3 |  100,000 |      300,000 |    36,022,848.50 |    120.0762
    4 |  100,000 |      400,000 |    48,028,117.25 |    120.0703
    5 |  100,000 |      500,000 |    60,011,762.84 |    120.0235
    6 |  100,000 |      600,000 |    72,036,466.16 |    120.0608
    7 |  100,000 |      700,000 |    84,039,572.76 |    120.0565
    8 |  100,000 |      800,000 |    96,067,808.29 |    120.0848
    9 |  100,000 |      900,000 |   108,140,485.43 |    120.1561
   10 |  100,000 |    1,000,000 |   120,132,024.27 |    120.1320


## Step 4: Final result

In [5]:
average = total_amount / total_records
print(f"Total records             : {total_records:,}")
print(f"Total transaction amount  : {total_amount:,.2f}")
print(f"Average transaction value : {average:.4f}")

Total records             : 1,000,000
Total transaction amount  : 120,132,024.27
Average transaction value : 120.1320


## Step 5: Verification (only possible because this demo file fits in RAM)

In [6]:
full = pd.read_csv(csv_path, usecols=["amount"])["amount"]
print(f"Full-load sum : {full.sum():,.2f} | mean : {full.mean():.4f}")

Full-load sum : 120,132,024.27 | mean : 120.1320


## Step 6: Clean up the temporary file

In [7]:
shutil.rmtree(tmp_dir)
print("Temporary files removed.")

Temporary files removed.
